# Inferencia CPU ? Project AmarketBoundingBoxes

Demo reproducible del modelo gobernante **Experimento A ? AMARKET-only**.

El notebook verifica el peso congelado, fuerza CPU, toma cinco im?genes determin?sticas del split `test` configurado y visualiza las detecciones.


In [ ]:
import hashlib
import os
import sys
from pathlib import Path

import yaml

os.environ['CUDA_VISIBLE_DEVICES'] = ''

def find_repo_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'src' / 'predict.py').is_file():
            return candidate
    raise RuntimeError('No se encontr? la ra?z del repositorio')

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))

MODEL_PATH = Path(os.environ.get(
    'AMARKET_MODEL_PATH',
    ROOT / 'outputs' / 'demo_model' / 'amarket_yolo11n_g4_A_best.pt',
))

DATASET_YAML = ROOT / 'configs' / 'dataset.yaml'

with DATASET_YAML.open(encoding='utf-8') as handle:
    dataset = yaml.safe_load(handle)

dataset_root = Path(dataset['path'])
if not dataset_root.is_absolute():
    dataset_root = ROOT / dataset_root

TEST_DIR = dataset_root / dataset['test']
OUTPUT_DIR = ROOT / 'outputs' / 'notebook_cpu_demo'

EXPECTED_SHA256 = (
    '9149dfef3093ca12a80fbb85b860822a'
    'e52d6ca5d8e2f1a6224211f92acd6712'
)

assert MODEL_PATH.is_file(), MODEL_PATH
assert TEST_DIR.is_dir(), TEST_DIR

print('ROOT=', ROOT)
print('MODEL_PATH=', MODEL_PATH)
print('TEST_DIR=', TEST_DIR)


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

actual_sha = sha256_file(MODEL_PATH)
assert actual_sha == EXPECTED_SHA256
print('MODEL_SHA256=PASS')
print(actual_sha)


In [ ]:
import torch

checkpoint = torch.load(
    MODEL_PATH,
    map_location='cpu',
    weights_only=False,
)

assert checkpoint is not None
del checkpoint

print('TORCH_LOAD_MAP_LOCATION_CPU=PASS')
print('CUDA_VISIBLE_DEVICES=', os.environ['CUDA_VISIBLE_DEVICES'])


In [ ]:
from src.predict import ejecutar_inferencia

summary = ejecutar_inferencia(
    MODEL_PATH,
    TEST_DIR,
    OUTPUT_DIR,
    device='cpu',
    conf=0.25,
    limit=5,
    overwrite=True,
)

assert summary['device'] == 'cpu'
assert summary['model_sha256'] == EXPECTED_SHA256
assert summary['n_images'] == 5

print('CPU_INFERENCE=PASS')
print('IMAGES=', summary['n_images'])
print(f"MEAN_INFERENCE_MS={summary['mean_inference_ms']:.3f}")


In [ ]:
from PIL import Image

annotated = sorted((OUTPUT_DIR / 'annotated').glob('*'))
assert len(annotated) == 5

try:
    get_ipython
except NameError:
    running_in_notebook = False
else:
    running_in_notebook = True

if running_in_notebook:
    from IPython.display import display

    for image_path in annotated:
        with Image.open(image_path) as image:
            display(image.convert('RGB').copy())
else:
    for image_path in annotated:
        print('ANNOTATED_IMAGE=', image_path)

print('VISUALIZATION_READY=PASS')


## Resultado final sobre TEST AMARKET

| M?trica | Resultado |
|---|---:|
| Precision | 1.000000 |
| Recall | 1.000000 |
| F1 | 1.000000 |
| mAP@0.5 | 0.995000 |
| mAP@0.5:0.95 | 0.992875 |

Test: **98 im?genes / 98 ground truths**.  
Modelo: **Experimento A ? AMARKET-only**.  
Evaluaci?n final: **CPU, sin tuning posterior sobre test**.

El test can?nico no produjo errores naturales a `conf=0.25` e `IoU=0.5`; esa brecha se reporta expl?citamente y no se fabricaron errores despu?s de consultar test.
